In [ ]:
# =============================================
# CELL 1: Import thư viện và cấu hình chung
# =============================================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from skimage import color, transform
from skimage.feature import hog

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Cấu hình chung ---
CLASS_NAMES  = ['Canada', 'Japan', 'Korea', 'USA', 'Vietnam']
CLASS_COLORS = ['red', 'orange', 'green', 'blue', 'purple']
IMG_SIZE     = (96, 96)

print('Thư viện đã sẵn sàng!')
print('Các lớp:', CLASS_NAMES)

In [ ]:
# =============================================
# CELL 2: Đọc ảnh và trích xuất đặc trưng HOG
# =============================================

def extract_hog_features(img_path, img_size=(96, 96)):
    """Đọc 1 ảnh, resize, chuyển xám, trích HOG features."""
    img = plt.imread(img_path)
    # Chuẩn hóa về float [0, 1]
    if img.dtype == np.uint8:
        img = img.astype(np.float32) / 255.0
    # Bỏ kênh alpha nếu có (RGBA -> RGB)
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    # Chuyển ảnh xám thành RGB nếu cần
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    # Resize về kích thước chuẩn
    img = transform.resize(img, img_size, anti_aliasing=True)
    # Chuyển sang ảnh xám để trích HOG
    gray = color.rgb2gray(img)
    # Trích xuất đặc trưng HOG
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True
    )
    return features, img  # trả về features và ảnh đã resize


# --- Duyệt qua tất cả thư mục và đọc ảnh ---
X_raw, y_raw, file_info = [], [], []

for idx, class_name in enumerate(CLASS_NAMES):
    folder_path = os.path.join('images', class_name)
    if not os.path.exists(folder_path):
        print(f'[!] Không tìm thấy thư mục: {folder_path}')
        continue
    files = sorted([f for f in os.listdir(folder_path)
                    if f.lower().endswith(('.bmp', '.png', '.jpg', '.jpeg'))])
    for fname in files:
        fpath = os.path.join(folder_path, fname)
        features, _ = extract_hog_features(fpath, IMG_SIZE)
        X_raw.append(features)
        y_raw.append(idx)
        file_info.append((class_name, fname, fpath))
    print(f'  [{class_name}]: {len(files)} ảnh')

X_raw = np.array(X_raw)
y_raw = np.array(y_raw)

print(f'\nTổng số ảnh    : {X_raw.shape[0]}')
print(f'Số đặc trưng HOG: {X_raw.shape[1]}')
print(f'Phân bố lớp    : {np.bincount(y_raw)}')

In [ ]:
# =============================================
# CELL 3: Minh họa đặc trưng HOG
# =============================================

fig, axes = plt.subplots(2, len(CLASS_NAMES), figsize=(16, 6))
fig.suptitle('Minh họa đặc trưng HOG cho từng lớp tiền tệ', fontsize=14, fontweight='bold')

for idx, class_name in enumerate(CLASS_NAMES):
    folder_path = os.path.join('images', class_name)
    files = sorted([f for f in os.listdir(folder_path)
                    if f.lower().endswith(('.bmp', '.png', '.jpg', '.jpeg'))])
    if not files:
        continue
    fpath = os.path.join(folder_path, files[0])
    img = plt.imread(fpath)
    if img.dtype == np.uint8:
        img = img.astype(np.float32) / 255.0
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = transform.resize(img, IMG_SIZE, anti_aliasing=True)
    gray = color.rgb2gray(img)
    _, hog_image = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), visualize=True, feature_vector=True)
    axes[0][idx].imshow(img)
    axes[0][idx].set_title(class_name, fontweight='bold')
    axes[0][idx].axis('off')
    axes[1][idx].imshow(hog_image, cmap='gray')
    axes[1][idx].set_title('HOG', color='gray')
    axes[1][idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# CELL 4: Chia tập train/test và huấn luyện SVM
# =============================================

# Bước 1: Chuẩn hóa đặc trưng (đưa về cùng thang đo)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Bước 2: Chia tập train (80%) và test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw  # giữ tỉ lệ lớp đều nhau
)
print(f'Tập train : {len(X_train)} ảnh')
print(f'Tập test  : {len(X_test)} ảnh')

# Bước 3: Huấn luyện SVM với kernel RBF
svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
print('\nĐang huấn luyện SVM...')
svm.fit(X_train, y_train)
print('Huấn luyện xong!')

# Bước 4: Đánh giá trên tập test
y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\nĐộ chính xác trên tập test: {acc * 100:.2f}%')
print('\nBáo cáo chi tiết:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# =============================================
# CELL 5: Ma trận nhầm lẫn (Confusion Matrix)
# =============================================

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Ma trận nhầm lẫn (Confusion Matrix)', fontsize=13, fontweight='bold')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# CELL 6: Dự đoán và hiển thị từng ảnh gốc
# =============================================

print('=== DỰ ĐOÁN TỪNG ẢNH GỐC ===')
total_ok, total_wrong = 0, 0

for idx, class_name in enumerate(CLASS_NAMES):
    folder_path = os.path.join('images', class_name)
    files = sorted([f for f in os.listdir(folder_path)
                    if f.lower().endswith(('.bmp', '.png', '.jpg', '.jpeg'))])
    class_ok, class_wrong = 0, 0
    print(f'\n===== {class_name} ({len(files)} ảnh) =====')

    for fname in files:
        fpath = os.path.join(folder_path, fname)
        features, img_display = extract_hog_features(fpath, IMG_SIZE)
        features_scaled = scaler.transform([features])

        probs     = svm.predict_proba(features_scaled)[0]
        pred_idx  = int(np.argmax(probs))
        predicted = CLASS_NAMES[pred_idx]

        if predicted == class_name:
            status = '[OK   ]'; class_ok += 1; total_ok += 1
        else:
            status = '[WRONG]'; class_wrong += 1; total_wrong += 1

        conf_str = f"{probs[pred_idx] * 100:.1f}%"
        print(f'  {status} {fname:20s} -> {predicted}  (Confidence: {conf_str})')

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        title_color = 'green' if predicted == class_name else 'red'
        fig.suptitle(f'[{class_name}] {fname}  ->  {predicted} {status}',
                     fontsize=12, fontweight='bold', color=title_color)
        axes[0].imshow(img_display)
        axes[0].axis('off')
        axes[0].set_title('Ảnh đầu vào')
        axes[1].bar(CLASS_NAMES, probs, color=CLASS_COLORS)
        axes[1].set_ylim(0, 1.15)
        axes[1].set_xlabel('Lớp')
        axes[1].set_ylabel('Xác suất')
        axes[1].set_title('SVM Probabilities (HOG Features)')
        for i, v in enumerate(probs):
            axes[1].text(i, v + 0.02, f'{v * 100:.1f}%', ha='center', va='bottom', fontsize=10)
        plt.tight_layout()
        plt.show()

    print(f'  >> {class_name}: OK={class_ok}, WRONG={class_wrong}')

total = total_ok + total_wrong
print('\n==============================')
print('TỔNG KẾT:')
print(f'  Tổng ảnh : {total}')
print(f'  OK       : {total_ok}  ({total_ok / total * 100:.1f}%)')
print(f'  WRONG    : {total_wrong}  ({total_wrong / total * 100:.1f}%)')
print(f'  Accuracy : {total_ok / total * 100:.2f}%')
print('==============================')